In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))

## LLM Initialization

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5-nano")

## 문서 처리

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

docs = [WebBaseLoader(url).load() for url in urls]
len(docs)

In [ ]:
docs_list = [item for sublist in docs for item in sublist]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=100
)

doc_splits = text_splitter.split_documents(docs_list)
len(doc_splits)

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, 
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [ ]:
from langchain.tools import tool

@tool
def retrieve_blog_posts(query: str) -> str:
    """Lilian Weng의 블로그 게시물에 대해 검색하고 정보를 알려주세요."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

retriever_tool = retrieve_blog_posts

- 도구 테스트

In [ ]:
retriever_tool.invoke({"query":"리워드 해킹의 타입"})

## 쿼리 생성

In [ ]:
from langgraph.graph import MessagesState
from langchain.chat_models import init_chat_model

response_model = init_chat_model("gpt-5-mini", temperature=0)

# 입력 메시지를 기반으로, 리트리버 도구를 사용하여 메시지를 가져올지, 
# 아니면 사용자에게 직접 응답할지 결정합니다
def generate_query_or_respond(state: MessagesState):
    """현재 상태를 바탕으로 답변을 생성하기 위해 모델을 호출합니다. 
    입력된 질문에 따라, 리트리버(retriever) 도구를 사용하여 정보를 검색할지, 
    아니면 사용자에게 즉시 답변할지를 결정합니다."""
    response = (
        response_model
        .bind_tools([retriever_tool]).invoke(state["messages"])  
    )
    return {"messages": [response]}

In [ ]:
input = {"messages": [{"role": "user", "content": "hello!"}]}
generate_query_or_respond(input)["messages"][-1].pretty_print()

In [ ]:
input = {
    "messages": [
        {
            "role": "user",
            "content": "Lilian Weng은 리워드(reward)의 유형에 대해 무엇이라고 말하나요?",
        }
    ]
}
generate_query_or_respond(input)["messages"][-1].pretty_print()

# 문서 등급 매기기

In [ ]:
GRADE_PROMPT = (
"""당신은 검색된 문서와 사용자 질문 간의 관련성을 평가하는 평가관입니다. \n 
검색된 문서 내용:  
{context}

 
사용자 질문: {question} 

문서가 사용자 질문과 관련된 키워드나 의미적 내용을 포함하고 있다면, '관련 있음'으로 평가하세요. \n
문서가 질문과 관련이 있는지 여부를 나타내기 위해 'yes' 또는 'no'로 이진 점수를 부여하세요."""
)

In [ ]:
from pydantic import BaseModel, Field

class GradeDocuments(BaseModel):  
    """관련성 체크를 위해 이진 점수(binary score)를 사용하여 문서를 채점합니다."""
    binary_score: str = Field(
        description="관련성 점수: 질문과 관련이 있으면 'yes', 관련이 없으면 'no'"
    )

In [ ]:
from typing import Literal

grader_model = init_chat_model("gpt-5-nano-2025-08-07", temperature=0)

def grade_documents(
    state: MessagesState,
) -> Literal["generate_answer", "rewrite_question"]:
    """검색된 문서들이 질문과 관련이 있는지 판단합니다."""
    question = state["messages"][0].content
    context = state["messages"][-1].content

    prompt = GRADE_PROMPT.format(question=question, context=context)
    response = (
        grader_model
        .with_structured_output(GradeDocuments).invoke(  
            [{"role": "user", "content": prompt}]
        )
    )
    score = response.binary_score

    if score == "yes":
        return "generate_answer"
    else:
        return "rewrite_question"

In [ ]:
from langchain_core.messages import convert_to_messages

# 과거의 대화 내역(State)을 강제로 주입하기 위한 입력 데이터 구조를 만드는 과정
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "Lilian Weng은 리워드 해킹(reward hacking)의 유형에 대해 무엇이라고 말하나요?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "리워드 해킹의 타입들"},
                    }
                ],
            },
            {"role": "tool", "content": "야옹", "tool_call_id": "1"},
        ]
    )
}
grade_documents(input)

# 관련성 없는 질문을 던져서 확인

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "Lilian Weng은 리워드 해킹(reward hacking)의 유형에 대해 무엇이라고 말하나요?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "리워드 해킹의 타입들"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": """보상 해킹은 두 가지 유형으로 분류될 수 있습니다: 
                환경 또는 목표의 잘못된 설정(misspecification), 그리고 보상 조작(tampering)입니다.""",
                "tool_call_id": "1",
            },
        ]
    )
}
grade_documents(input)

## 질문 재작성

In [ ]:
from langchain.messages import HumanMessage

REWRITE_PROMPT = (
"입력 내용을 살펴보고 그 안에 담긴 근본적인 의도와 의미를 분석하세요.\n" 
"사용자의 초기 질문입니다:" 
"\n ------- \n" 
"{question}" 
"\n ------- \n" 
"개선된 질문을 작성하세요:"
)


def rewrite_question(state: MessagesState):
    """Rewrite the original user question."""
    messages = state["messages"]
    question = messages[0].content
    prompt = REWRITE_PROMPT.format(question=question)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [HumanMessage(content=response.content)]}

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "Lilian Weng은 리워드 해킹(reward hacking)의 유형에 대해 무엇이라고 말하나요?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "리워드 해킹의 타입들"},
                    }
                ],
            },
            {"role": "tool", "content": "야옹", "tool_call_id": "1"},
        ]
    )
}

response = rewrite_question(input)
print(response["messages"][-1].content)

## 최종 답변 생성

In [ ]:
GENERATE_PROMPT = (
"""당신은 질의응답 작업을 수행하는 어시스턴트입니다. 
검색된 다음의 정보들을 사용하여 질문에 답하세요. 
"답을 모른다면, 단순히 모른다고 답변하세요. 
최대 세 문장 내로 답변하고 간결함을 유지하세요.\n
질문: {question} \n" 
참조 정보: {context}"""
)


def generate_answer(state: MessagesState):
    """Generate an answer."""
    question = state["messages"][0].content
    context = state["messages"][-1].content
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [response]}

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "Lilian Weng은 리워드 해킹(reward hacking)의 유형에 대해 무엇이라고 말하나요?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "리워드 해킹의 타입들"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": """
                "보상 해킹은 두 가지 유형으로 분류될 수 있습니다: 
                환경 또는 목표의 잘못된 설정(misspecification), 그리고 보상 변조(tampering)입니다."
                """,
                "tool_call_id": "1",
            },
        ]
    )
}

response = generate_answer(input)
response["messages"][-1].pretty_print()

## 그래프 구성

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

workflow = StateGraph(MessagesState)

# Define the nodes we will cycle between
workflow.add_node(generate_query_or_respond)
workflow.add_node("retrieve", ToolNode([retriever_tool]))
workflow.add_node(rewrite_question)
workflow.add_node(generate_answer)

workflow.add_edge(START, "generate_query_or_respond")

# Decide whether to retrieve
workflow.add_conditional_edges(
    "generate_query_or_respond",
    # Assess LLM decision (call `retriever_tool` tool or respond to the user)
    tools_condition,
    {
        # Translate the condition outputs to nodes in our graph
        "tools": "retrieve",
        END: END,
    },
)

# Edges taken after the `action` node is called.
workflow.add_conditional_edges(
    "retrieve",
    # Assess agent decision
    grade_documents,
)
workflow.add_edge("generate_answer", END)
workflow.add_edge("rewrite_question", "generate_query_or_respond")

# Compile
graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Run the agentic RAG

In [ ]:
for chunk in graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            }
        ]
    }
):
    for node, update in chunk.items():
        print("Update from node", node)
        update["messages"][-1].pretty_print()
        print("\n\n")